# Adjusting the branch a split creates

A flow can fan out into a property the source does not have, via `split=`.
That property is not on the source, so a bare trait in `where=` has to mean
the destination branch.

The claim: `Multiply(3, where=severity["severe"])` on a split into
`mild` / `severe` scales only the severe inflow, and matches both
`Dest(severity["severe"])` and the split weights. Severe should be
nine times mild (weight 0.75 times 3, against weight 0.25).


## Severe versus mild

One susceptible person moves into infectious at rate 1. The split sends
a quarter to mild and three quarters to severe, and the severe branch is
then multiplied by 3. The next cell plots the two inflows. The bare
`where=`, an explicit `Dest(...)`, and the formula should overlie.


In [ ]:
from typing import Any

import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import (
    Dest,
    FlowModel,
    Multiply,
    Property,
    PropertyMap,
    TransitionFlow,
)

state = Property("state", ("S", "I"))
severity = Property("severity", ("mild", "severe"))
pmap = PropertyMap.from_property(state).stratify(severity, where=state["I"])
split = {severity: {"mild": 0.25, "severe": 0.75}}


def inflows(where: Any) -> dict[str, float]:
    """People per unit time arriving in each severity, from one susceptible."""
    model = FlowModel(pmap)
    model.add_flow(
        TransitionFlow(
            "inf",
            state["S"],
            state["I"],
            1.0,
            split=split,
            adjust=[Multiply(3.0, where=where)],
        )
    )
    compiled = model.compile()
    dy = np.asarray(compiled.vector_field(0.0, np.ones(pmap.size), {}))
    return {
        name: float(dy[pmap.select(state["I"] & severity[name])][0])
        for name in severity.traits
    }


bare = inflows(severity["severe"])
dest = inflows(Dest(severity["severe"]))
formula = {"mild": 0.25, "severe": 0.75 * 3.0}
np.testing.assert_allclose(
    [bare[name] for name in severity.traits],
    [formula[name] for name in severity.traits],
)
np.testing.assert_allclose(
    [bare[name] for name in severity.traits],
    [dest[name] for name in severity.traits],
)

pd.DataFrame({"bare where": bare, "Dest(...)": dest, "formula": formula}).plot(
    kind="bar",
    title="Severe inflow is the split weight times 3; mild is the weight alone",
    labels={"index": "destination severity", "value": "people per unit time"},
)
